In [ ]:
import pandas as pd
import geopandas as gpd

# Lista dos 30 municípios da 15ª RS
municipios_15rs = [
    'Ângulo', 'Astorga', 'Atalaia', 'Colorado', 'Doutor Camargo',
    'Floraí', 'Floresta', 'Flórida', 'Iguaraçu', 'Itaguajé',
    'Itambé', 'Ivatuba', 'Lobato', 'Mandaguaçu', 'Mandaguari',
    'Marialva', 'Maringá', 'Munhoz de Melo', 'Nossa Senhora das Graças',
    'Nova Esperança', 'Ourizona', 'Paiçandu', 'Paranacity',
    'Presidente Castelo Branco', 'Santa Fé', 'Santa Inês',
    'Santo Inácio', 'São Jorge do Ivaí', 'Sarandi', 'Uniflor'
]

# Buscar códigos no shapefile
gdf = gpd.read_file('/home/valentim/divea/data/gis/Pr_Municipios_2024/PR_Municipios_2024.shp')
gdf['CD_MUN6'] = gdf['CD_MUN'].str[:6]

muns_15rs = gdf[gdf['NM_MUN'].isin(municipios_15rs)][['CD_MUN6', 'NM_MUN']]
print(f"Encontrados: {len(muns_15rs)} de 30")
print(muns_15rs.sort_values('NM_MUN').to_string(index=False))

In [ ]:
# Salvar lista de municípios da 15ª RS
muns_15rs.to_csv('/home/valentim/divea/data/processed/municipios_15rs.csv', index=False)
print("Salvo.")

# Verificar se estão nos dados do SIVEP
df_pr = pd.read_parquet('/home/valentim/divea/data/processed/sivep_parana.parquet')
codigos_15rs = muns_15rs['CD_MUN6'].tolist()
df_15rs = df_pr[df_pr['CO_MUN_RES'].isin(codigos_15rs)]

print(f"\nRegistros 15ª RS: {len(df_15rs):,}")
print(f"Período: {df_15rs['ANO_ARQUIVO'].min()} a {df_15rs['ANO_ARQUIVO'].max()}")
print("\nCasos por ano:")
print(df_15rs['ANO_ARQUIVO'].value_counts().sort_index())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Salvar dataset 15ª RS
df_15rs['DT_NOTIFIC'] = pd.to_datetime(df_15rs['DT_NOTIFIC'], dayfirst=True, errors='coerce')
df_15rs.to_parquet('/home/valentim/divea/data/processed/sivep_15rs.parquet', index=False)
print("Salvo.")

# Série temporal semanal
serie_15rs = (
    df_15rs.groupby(df_15rs['DT_NOTIFIC'].dt.to_period('W'))
    .size()
    .reset_index(name='casos')
)
serie_15rs['DT_NOTIFIC'] = serie_15rs['DT_NOTIFIC'].dt.to_timestamp()
serie_15rs = serie_15rs.set_index('DT_NOTIFIC')

# Séries por vírus
def serie_virus(df, coluna, valor='1', nome=''):
    s = (
        df[df[coluna] == valor]
        .groupby(df['DT_NOTIFIC'].dt.to_period('W'))
        .size()
        .reset_index(name=nome)
    )
    s['DT_NOTIFIC'] = s['DT_NOTIFIC'].dt.to_timestamp()
    return s.set_index('DT_NOTIFIC')

s_flu = serie_virus(df_15rs, 'POS_PCRFLU', nome='influenza')
s_cov = serie_virus(df_15rs, 'PCR_SARS2', nome='covid')
s_vsr = serie_virus(df_15rs, 'PCR_VSR', nome='vsr')

df_series_15rs = serie_15rs.join([s_flu, s_cov, s_vsr]).fillna(0)
df_series_15rs.to_parquet('/home/valentim/divea/data/processed/series_semanais_15rs.parquet')

print(f"Shape: {df_series_15rs.shape}")
print(df_series_15rs.head())

df_15rs = df_pr[df_pr['CO_MUN_RES'].isin(codigos_15rs)].copy()

In [ ]:
# Canal endêmico por semana epidemiológica
# Usa média e desvio padrão histórico excluindo anos pandêmicos

df_15rs_hist = df_15rs[
    (df_15rs['ANO_ARQUIVO'] != '2020') & 
    (df_15rs['ANO_ARQUIVO'] != '2021')
].copy()

# Agrupar por semana epidemiológica
df_15rs_hist['SEM_NOT'] = pd.to_numeric(df_15rs_hist['SEM_NOT'], errors='coerce')
df_15rs_hist['ANO_ARQUIVO'] = pd.to_numeric(df_15rs_hist['ANO_ARQUIVO'], errors='coerce')

canal = (
    df_15rs_hist.groupby('SEM_NOT')
    .size()
    .reset_index(name='casos')
)

# Calcular por ano e semana para ter variabilidade
casos_por_ano_sem = (
    df_15rs_hist.groupby(['ANO_ARQUIVO', 'SEM_NOT'])
    .size()
    .reset_index(name='casos')
)

canal_endemico = (
    casos_por_ano_sem.groupby('SEM_NOT')['casos']
    .agg(['mean', 'std'])
    .reset_index()
)
canal_endemico.columns = ['semana', 'media', 'std']
canal_endemico['limite_inferior'] = (canal_endemico['media'] - canal_endemico['std']).clip(lower=0)
canal_endemico['limite_superior'] = canal_endemico['media'] + canal_endemico['std']
canal_endemico['alerta'] = canal_endemico['media'] + 2 * canal_endemico['std']
canal_endemico['epidemia'] = canal_endemico['media'] + 3 * canal_endemico['std']

print(canal_endemico.head(10))

In [ ]:
canal_endemico.to_parquet('/home/valentim/divea/data/processed/canal_endemico_15rs.parquet', index=False)

# Visualizar canal endêmico com dados de 2026
casos_2026 = (
    df_15rs[df_15rs['ANO_ARQUIVO'] == '2026']
    .groupby(pd.to_numeric(df_15rs[df_15rs['ANO_ARQUIVO'] == '2026']['SEM_NOT'], errors='coerce'))
    .size()
    .reset_index(name='casos')
)
casos_2026.columns = ['semana', 'casos']

fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(canal_endemico['semana'],
                canal_endemico['limite_inferior'],
                canal_endemico['limite_superior'],
                alpha=0.3, color='green', label='Zona segura')

ax.fill_between(canal_endemico['semana'],
                canal_endemico['limite_superior'],
                canal_endemico['alerta'],
                alpha=0.3, color='yellow', label='Zona de alerta')

ax.fill_between(canal_endemico['semana'],
                canal_endemico['alerta'],
                canal_endemico['epidemia'],
                alpha=0.3, color='orange', label='Zona de risco')

ax.plot(canal_endemico['semana'], canal_endemico['media'],
        color='green', linewidth=1.5, linestyle='--', label='Média histórica')

ax.plot(casos_2026['semana'], casos_2026['casos'],
        color='red', linewidth=2, label='2026')

ax.set_title('Canal Endêmico SRAG — 15ª RS Maringá (2026)')
ax.set_xlabel('Semana Epidemiológica')
ax.set_ylabel('Casos')
ax.legend()
ax.set_xlim(1, 52)
plt.tight_layout()
plt.savefig('/home/valentim/divea/outputs/canal_endemico_15rs.png', dpi=150)
plt.show()